In [4]:
import numpy as np

## 5V power sizing

In [5]:
V = 5
V_unreg = 12

consumers = []

# https://forum.pjrc.com/index.php?threads/what-is-the-power-consumption-of-the-teensy-3-6.47256/
i_teensy = 80e-3; consumers += [i_teensy]

# see page 37 of https://www.pololu.com/file/0J1663/TB9051FTG_datasheet_en_20190206.pdf
i_motor_driver = 5e-3; consumers += [i_motor_driver] # max rating

# https://www.pololu.com/product/4864
i_encoder_pololu = 10e-3; consumers += [i_encoder_pololu]

# https://www.sameskydevices.com/product/resource/amt10.pdf
i_encoder_samesky = 6e-3; consumers += [i_encoder_samesky]

# https://www.pololu.com/product/4040/specs
i_current_sens = 14e-3; consumers += [i_current_sens]

# any red led datasheet, for example https://www.farnell.com/datasheets/1498852.pdf
i_led = 20e-3; consumers += [i_led]

# switches with built-in teensy pullups, assuming 10k resistors - can't find exact spec in datasheet
i_switch = V / 10e3; consumers += 3*[i_switch]

i_total = np.array(consumers).sum()
p_total = V * i_total
p_dissip = (V_unreg - V) * i_total

print(f"Total current: {i_total:.3} A\nTotal power: {p_total:.2} W")

print(f"Power dissipation need by linear voltage regulator: {p_dissip:.2} W")

Total current: 0.137 A
Total power: 0.68 W
Power dissipation need by linear voltage regulator: 0.96 W


### Heat sink linear regulator

In [6]:
Ta_max = 38 # C

# From example A in Boyd catalog here https://info.boydcorp.com/hubfs/Thermal/Air-Cooling/Boyd-Board-Level-Heatsinks-Catalog.pdf
Rcs = 0.5 # C/W 
# from LP2954 datasheet
Rjc = 0.9 # C/W
Tj_max = 125 # C

Rsa_max = (Tj_max - Ta_max) / p_dissip - (Rjc + Rcs)
Rsa_max # PF730G from Boyd catalog could work

np.float64(89.65180533751962)

### No heat sink linear regulator

In [11]:
# from LP2954 datasheet
"""
Thermal resistance value RθJA is based on the EIA/JEDEC High-K printed circuit board defined by 
JESD51-7 - High Effective Thermal Conductivity Test Board for Leaded Surface Mount Packages. 
The TO-220 (NDE) package is vertically mounted in center of JEDEC High-K test board (JESD 51-7) with no additional heat sink. 
This is a through-hole package; this is NOT a surface mount package.
"""
Rja = 80.3 # C/W

# since my board does not reselmble High-K test board, realistic value is more like this
Rja = 100 # C/W

Tj = Ta_max + Rja * p_dissip
assert Tj > Tj_max

So the conclusion is we either need a heat sink or a switching regulator (for example LM2574-5.0YN on DigiKey)